# Structure Inspection: PDF Image & JSON Side-by-Side

This notebook allows you to visually inspect the extracted table structures by displaying the PDF page image (left) and the corresponding JSON file (right) for each example.

Use the dropdown below to select a page.

In [1]:
import re
import json
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

# Directory containing images and jsons
IMG_DIR = Path('/home/pwiesenbach/CardioGuidelinesGraph/src/data/guidelines/structures/from_pdf_images/pdf_pages')
JSON_DIR = IMG_DIR

def extract_num(filename):
    match = re.match(r'_([0-9]+)_page_000\.png', filename)
    return int(match.group(1)) if match else float('inf')

# Find all pairs (_N_page_000.png and _N.json)
pairs = []
for img_file in IMG_DIR.glob('_*_page_000.png'):
    base = img_file.stem.split('_page_')[0]
    json_file = JSON_DIR / f'{base}.json'
    if json_file.exists():
        pairs.append((img_file, json_file))

# Sort pairs by numeric prefix
pairs_sorted = sorted(pairs, key=lambda x: extract_num(x[0].name))
options = [f'{img.name} | {json.name}' for img, json in pairs_sorted]
pair_map = {opt: (img, json) for opt, (img, json) in zip(options, pairs_sorted)}

dropdown = widgets.Dropdown(
    options=options,
    value=options[0] if options else None,
    description='Select Pair:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
    )

output_area = widgets.Output()

def show_pair(change):
    with output_area:
        output_area.clear_output(wait=True)
        if not change['new']:
            print('No pair selected.')
            return
        img_path, json_path = pair_map[change['new']]

        # Read image data for ipywidgets.Image
        try:
            with open(img_path, 'rb') as img_file:
                img_bytes = img_file.read()
            img_widget = widgets.Image(value=img_bytes, format='png', layout=widgets.Layout(width='100%', height='auto'))
        except Exception as e:
            img_widget = widgets.HTML(f'<p>Error loading image: {e}</p>')

        # Read JSON data
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            json_str = json.dumps(json_data, indent=2, default=str)
        except Exception as e:
            json_str = f'Error loading JSON: {e}'

        # Create side-by-side layout using ipywidgets, each half width
        left_box = widgets.VBox([widgets.HTML(f"<h3>PDF Page Image</h3><p style='font-size:12px;'>{img_path.name}</p>"), img_widget], layout=widgets.Layout(width='50%'))
        right_box = widgets.VBox([widgets.HTML(f"<h3>JSON Structure</h3><p style='font-size:12px;'>{json_path.name}</p>"), widgets.HTML(f"<pre style='overflow-x:auto; max-height:600px; font-size:11px; white-space:pre-wrap;'>{json_str}</pre>")], layout=widgets.Layout(width='50%'))
        hbox = widgets.HBox([left_box, right_box], layout=widgets.Layout(width='100%'))
        display(hbox)

dropdown.observe(show_pair, names='value')
display(dropdown)
display(output_area)
# Show initial pair
if dropdown.value:
    show_pair({'new': dropdown.value})

Dropdown(description='Select Pair:', layout=Layout(width='500px'), options=('_2_page_000.png | _2.json', '_3_p…

Output()